# Task 4.1a. Baseline không dùng graph

| | |
|---|---|
| **Mục đích** | Mức sàn để so sánh với GNN: majority, ngẫu nhiên theo phân bố lớp, và TF-IDF cộng hồi quy logistic có trọng số lớp (token-method, token-nodes) |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Joern Code Graphs` (chỉ cần `graph_a`) |
| **Đầu ra** | `results_4_1a/`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `summary.md` |
| **Máy chạy** | CPU, bật Internet. 2 protocol, 3 seed |

In [1]:
import os
import subprocess
import sys
from pathlib import Path

GIT_BRANCH = "main"
REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
REPO_DIR = Path("/tmp/g41a/software-bug-detection-using-graphs")  # Ngoài thư mục output
OUT_DIR = Path("/kaggle/working/results_4_1a")

if not Path("/kaggle/input").is_dir():
    raise RuntimeError("Notebook này chỉ chạy trên Kaggle.")
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_BRANCH, REPO_URL, str(REPO_DIR)],
                   check=True, timeout=180, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
else:
    subprocess.run(["git", "pull", "origin", GIT_BRANCH], cwd=REPO_DIR, check=True, timeout=180)


Cloning into '/tmp/g41a/software-bug-detection-using-graphs'...


## 1. Cấu hình và xác định dữ liệu đầu vào

In [2]:
candidates = [p for p in Path("/kaggle/input").rglob("graph_a") if p.is_dir()]
assert candidates, "Không thấy graph_a, hãy Add Input dataset DiverseVul 9077 - Joern Code Graphs."
GRAPH_A_DIR = candidates[0]
print("graph_a:", GRAPH_A_DIR, "-", sum(1 for _ in GRAPH_A_DIR.glob("*.json")), "tệp JSON")


graph_a: /kaggle/input/datasets/vtnmai/diversevul-9077-joern-code-graphs/graphs_subset_9077/subset_9077/graph_a - 9077 tệp JSON


## 2. Chạy baseline

In [3]:
cmd = [sys.executable, "scripts/run_baselines_4_1a.py",
       "--graph-a-dir", str(GRAPH_A_DIR), "--out-dir", str(OUT_DIR),
       "--seeds", "42", "43", "44"]
result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
print(result.stdout)
print(result.stderr, file=sys.stderr)
result.check_returncode()


Số mẫu hợp lệ đọc được: 7799
project_wise: train=6101, validation=762, test=936
seen_project: train=6272, validation=764, test=763
Đã ghi kết quả vào /kaggle/working/results_4_1a



## 3. Kết quả

Bảng trung bình ± độ lệch chuẩn theo seed. Tệp đầy đủ nằm trong `results_4_1a/`.

In [4]:
print((OUT_DIR / "summary.md").read_text(encoding="utf-8"))


# Kết quả baseline không dùng graph (Task 4.1a)

Giá trị là trung bình ± độ lệch chuẩn theo seed.

| Protocol | Split | Model | Macro-F1 | Weighted-F1 | MCC | Accuracy |
| :--- | :--- | :--- | ---: | ---: | ---: | ---: |
| project_wise | test | majority | 0.0100 ± 0.0000 | 0.0301 ± 0.0000 | 0.0000 ± 0.0000 | 0.1303 ± 0.0000 |
| project_wise | test | random | 0.0393 ± 0.0039 | 0.0777 ± 0.0079 | 0.0016 ± 0.0063 | 0.0755 ± 0.0059 |
| project_wise | test | token_method | 0.0875 ± 0.0000 | 0.1418 ± 0.0000 | 0.0727 ± 0.0000 | 0.1528 ± 0.0000 |
| project_wise | test | token_nodes | 0.1000 ± 0.0000 | 0.1495 ± 0.0000 | 0.0868 ± 0.0000 | 0.1656 ± 0.0000 |
| project_wise | validation | majority | 0.0206 ± 0.0000 | 0.1476 ± 0.0000 | 0.0000 ± 0.0000 | 0.3110 ± 0.0000 |
| project_wise | validation | random | 0.0394 ± 0.0041 | 0.0979 ± 0.0042 | -0.0014 ± 0.0048 | 0.0840 ± 0.0047 |
| project_wise | validation | token_method | 0.0504 ± 0.0000 | 0.1195 ± 0.0000 | 0.0445 ± 0.0000 | 0.1115 ± 0.0000 |
| pr